In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
import numpy as np
import torch
import os
import sys
sys.path.append("..")
from utils.balanced_builders_hdf import *
from models.autoencoder_classifier import *
from final_pipline.similarity_latent import *
from final_pipline.latent_similarty_funcs import *
from final_pipline.external_testset import *
from numpy.linalg import norm
from config import *

In [2]:
sg = build_balanced_sg_loaders_from_h5(
    h5_path=xrd_dataset,
    min_count_sg=20000,
    per_class_cap_sg=20000,
    batch_size=256,
    val_split=0.1,
    test_split=0.1,
    seed=42,
    num_workers=0,
)

print(sg["num_classes"])
print(sg["sizes"])

22
{'train': 352000, 'val': 44000, 'test': 44000}


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recreate model EXACTLY the same way
model = DeepConvAutoencoderClassifier(
    input_length=sg["input_len"],
    latent_dim=64,
    cls_dim=128,
    num_classes=sg["num_classes"],
    use_projection_head=True
).to(device)

# Load weights
checkpoint = torch.load(SG_Cls, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])  

model.eval()

✓ Model loaded successfully


C:\Users\doaam\AppData\Local\Temp\ipykernel_22556\1630458983.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(SG_Cls, map_location=device)


In [5]:
external_acc = evaluate_sg(model, sg["test_loader"], device, sg["num_classes"])
print("SG Test Accuracy:", external_acc * 100)

SG Test Accuracy: 95.825


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

train_latents, train_labels, train_indices = extract_latent_vectors(
    model,
    sg,
    device,
    loader_name="train_loader"
)

print("Train latents shape:", train_latents.shape)
print("Train labels shape:", train_labels.shape)
print("Train indices shape:", train_indices.shape)


Train latents shape: (352000, 128)
Train labels shape: (352000,)
Train indices shape: (352000,)


In [5]:

train_path = os.path.join(SG_LATENT , "xrd_train_latents_sg.npz")
np.savez_compressed(
   train_path,
    latents=train_latents,
    labels=train_labels,
    indices=train_indices,
    class_names=sg["class_names"]
)

In [ ]:
data = np.load(train_path, allow_pickle=True)
train_latents = data["latents"]
train_labels  = data["labels"]
train_indices = data["indices"]
class_names   = data["class_names"]
print("Loaded shapes:")
print("Latents:", train_latents.shape)
print("Labels:", train_labels.shape)
print("Indices:", train_indices.shape)
print("Classes:", class_names)


Loaded shapes:
Latents: (275968, 64)
Labels: (275968,)
Indices: (275968,)
Classes: [  1   2   8  11  12  14  15  19  38  61  62  63  71 123 139 164 166 187
 194 216 221 225 227]


In [6]:
test_latents, test_labels, test_indices = extract_latent_vectors(
    model,
    sg,
    device,
    loader_name="test_loader"
)

In [8]:
test_latents_path = os.path.join(SG_LATENT, "xrd_test_latents_sg.npz")

np.savez_compressed(
   test_latents_path,
    latents=test_latents,
    labels=test_labels,
    indices=test_indices,
    class_names=sg["class_names"]
)